# 从过程式编程到面向对象编程（OOP）

这份 Notebook 根据 `OOP approach.docx` 的知识主线重新整理。原文只作为学习材料；其中混合的 C、Java 和 Python 伪代码已统一改写为可运行的 Python。

完成后，你应该能够：

- 解释为什么要把重复代码提取为函数；
- 理解参数、局部变量和 `return`；
- 区分类、对象、属性、方法、`self` 和 `__init__`；
- 用代码说明封装、继承、抽象和多态；
- 判断一个问题更适合函数还是类。

## 使用方法与学习路线

先阅读每个代码单元格上方的问题并预测输出，再按 `Shift + Enter` 运行。学完后使用 **Kernel → Restart Kernel and Run All Cells**，检查整份 Notebook 是否能脱离旧状态独立运行。

```text
重复代码 → 函数 → 通用函数 → 类与对象 → 封装 → 继承 → 抽象 → 多态
```

In [ ]:
import sys

print(f"当前 Python 版本：{sys.version.split()[0]}")

## 0. 先读懂原文中的伪代码

原文的思想有教学价值，但代码不是某一种语言的完整程序。例如：

| 原文写法 | Python 写法或说明 |
|---|---|
| `this.power_order` | `self._power_order` |
| `class Child extend Parent` | `class Child(Parent):` |
| `result(...)` | `return ...` |
| `proctected` | 拼写错误；Python 也没有该访问修饰符 |
| `private` / `protected` | Python 通常用命名约定表达非公开成员 |
| `Sum_Power` / `Sum_Powers` | 名称不一致，应统一 |

原文还让函数修改全局变量 `c`，同时又在 `main` 中声明局部 `c`。这会造成作用域含糊；Python 示例将使用明确的返回值。

## 1. 从重复的过程式代码开始

任务：分别计算两组数字的平方和。运行前先预测两个结果。

In [ ]:
a1, b1 = 2, 3
result_1 = a1 * a1 + b1 * b1

a2, b2 = 4, 5
result_2 = a2 * a2 + b2 * b2

print("第一次平方和：", result_1)
print("第二次平方和：", result_2)

assert result_1 == 13
assert result_2 == 41

代码可以工作，但计算规则 `x * x + y * y` 被复制了。复制会带来两个问题：

1. 规则改变时，需要修改很多位置；
2. 某一处修改遗漏时，相同任务可能得到不一致的结果。

第一步不是立刻使用类，而是先把重复规则提取成函数。

### 为什么不建议用全局变量保存结果？

下面的函数每次都覆盖同一份 `latest_result`。这段代码是为了观察问题，不是推荐写法。

In [ ]:
latest_result = None

def save_sum_power(x, y, power_order):
    global latest_result
    latest_result = x ** power_order + y ** power_order


save_sum_power(2, 3, 2)
first_snapshot = latest_result

save_sum_power(2, 3, 3)
second_snapshot = latest_result

print("第一次调用的快照：", first_snapshot)
print("第二次调用的快照：", second_snapshot)
print("全局变量现在只保留：", latest_result)

assert first_snapshot == 13
assert second_snapshot == 35
assert latest_result == 35

全局变量是共享状态，任何函数都可能修改它。调用者还必须知道函数在背后修改了哪个名字。更清晰的方式是让函数使用局部变量，并通过 `return` 明确交回结果。

In [ ]:
def sum_squares(x, y):
    x_square = x * x
    y_square = y * y
    return x_square + y_square


def sum_cubes(x, y):
    return x ** 3 + y ** 3


def sum_power(x, y, power_order):
    """返回 x 和 y 的 power_order 次幂之和。"""
    if type(power_order) is not int or power_order < 1:
        raise ValueError("power_order 必须是正整数")
    return x ** power_order + y ** power_order


print("平方和：", sum_power(2, 3, 2))
print("立方和：", sum_power(2, 3, 3))
print("四次幂之和：", sum_power(2, 3, 4))

assert sum_squares(2, 3) == 13
assert sum_cubes(2, 3) == 35
assert sum_power(2, 3, 2) == 13
assert sum_power(2, 3, 3) == 35
assert sum_power(2, 3, 4) == 97
assert sum_power(2, 3, 5) == 275

### 第一阶段结论

这个小问题到这里其实已经解决得很好：

- 参数是函数的明确输入；
- 局部变量只属于本次调用；
- `return` 是明确输出；
- `power_order` 把多个专用函数推广成一个通用函数。

> **重要判断：OOP 不是所有问题的必选答案。** 如果只是偶尔进行一次无状态计算，`sum_power()` 通常就是最简单的设计。

## 2. 类与对象：让计算器记住自己的配置

如果程序会反复使用同一个幂次，我们可以创建“平方和计算器”和“立方和计算器”。每个对象都保存自己的 `power_order`。

In [ ]:
class SumPower:
    """保存幂次配置，并计算两个数的对应幂之和。"""

    def __init__(self, power_order):
        if type(power_order) is not int or power_order < 1:
            raise ValueError("power_order 必须是正整数")
        self._power_order = power_order

    @property
    def power_order(self):
        return self._power_order

    def _power(self, x):
        return x ** self._power_order

    def calculate(self, x, y):
        return self._power(x) + self._power(y)

运行前思考：下面两个对象调用的是同名方法，为什么结果不同？

In [ ]:
square_sum = SumPower(2)
cube_sum = SumPower(3)

print("square_sum 保存的幂次：", square_sum.power_order)
print("cube_sum 保存的幂次：", cube_sum.power_order)
print("平方和：", square_sum.calculate(2, 3))
print("立方和：", cube_sum.calculate(2, 3))

assert square_sum.power_order == 2
assert cube_sum.power_order == 3
assert square_sum.calculate(2, 3) == 13
assert cube_sum.calculate(2, 3) == 35

### 逐项解析

| 代码 | 含义 |
|---|---|
| `SumPower` | 类：对象的设计图 |
| `square_sum` | 对象，也叫实例 |
| `__init__` | 初始化方法；创建对象时自动调用 |
| `self` | 当前正在操作的对象 |
| `self._power_order` | 每个对象自己的状态 |
| `_power()` | 内部辅助方法 |
| `calculate()` | 外部调用的公共 API |

执行 `square_sum.calculate(2, 3)` 时，`self` 就是 `square_sum`，因此读取到的幂次是 `2`。对 `cube_sum` 调用时，`self` 换成另一个对象，读取到的幂次是 `3`。这里是**同一个类的不同实例保存了不同状态**；它与后文“子类重写方法后发生动态派发”的多态机制不同。

## 3. 封装（Encapsulation）

封装是把相关状态和行为放进同一个边界，并通过稳定的 API 维护合法状态。本例在初始化时拒绝非法幂次，并通过只读属性公开 `power_order`。

Python 中的单下划线 `_power_order` 和 `_power()` 只是“内部使用”的约定，不是绝对访问限制；`@property` 没有 setter，表示不能通过公共属性赋值，但外部技术上仍能直接碰到 `_power_order`。因此，Python 的封装主要依靠清晰 API 和开发者约定，而不是安全隔离。

In [ ]:
try:
    square_sum.power_order = 4
except AttributeError:
    print("[通过] power_order 没有公共 setter。")
else:
    raise AssertionError("power_order 本应是只读属性")

for invalid_order in (0, -1, 2.5, "3", True):
    try:
        SumPower(invalid_order)
    except ValueError:
        pass
    else:
        raise AssertionError(f"没有拒绝非法幂次：{invalid_order!r}")

assert square_sum.power_order == 2
print("[通过] 非法幂次均被拒绝，对象状态仍然有效。")

## 4. 继承（Inheritance）

现在创建一种带名称、会生成说明文字的幂次计算器。它仍然**是一种** `SumPower`，因此可以继承父类。

In [ ]:
class NamedSumPower(SumPower):
    def __init__(self, power_order, name):
        super().__init__(power_order)
        self.name = name

    def describe(self, x, y):
        result = self.calculate(x, y)
        return f"{self.name}（{self.power_order} 次幂）：{result}"


named_cube_sum = NamedSumPower(3, "立方和")

print(named_cube_sum.describe(2, 3))

assert isinstance(named_cube_sum, SumPower)
assert named_cube_sum.calculate(2, 3) == 35
assert named_cube_sum.describe(2, 3) == "立方和（3 次幂）：35"

解析：

- `NamedSumPower(SumPower)` 声明父子关系；
- `super().__init__(power_order)` 复用父类的初始化与校验；
- 子类自动获得 `calculate()`，再增加自己的 `name` 和 `describe()`。

继承应表达合理的 **is-a（是一种）** 关系。仅仅为了少复制几行代码，并不足以证明应该继承。

## 5. 抽象与多态（Abstraction & Polymorphism）

另一种需求是：不同计算器可能用完全不同的算法，但外部都希望调用 `calculate(x, y)`。抽象父类规定共同能力，把具体的 `_power()` 实现交给子类。

In [ ]:
from abc import ABC, abstractmethod


class AbstractPowerSum(ABC):
    @abstractmethod
    def _power(self, x):
        """子类必须实现具体的幂运算。"""
        raise NotImplementedError

    def calculate(self, x, y):
        return self._power(x) + self._power(y)


class SquareSum(AbstractPowerSum):
    def _power(self, x):
        return x * x


class CubeSum(AbstractPowerSum):
    def _power(self, x):
        return x * x * x

运行前预测：循环中的调用形式完全相同，输出为什么会是两个不同的数字？

In [ ]:
calculators = [SquareSum(), CubeSum()]
results = []

for calculator in calculators:
    result = calculator.calculate(2, 3)
    results.append(result)
    print(type(calculator).__name__, "→", result)

assert results == [13, 35]

这里的多态不是“只要方法重名就算完成”，而是运行时的动态派发：

```text
calculator.calculate(2, 3)
        ↓
AbstractPowerSum.calculate()       # 父类中同一份实现
        ↓
self._power(2) + self._power(3)   # 根据 self 的实际类型选择方法
   ├─ SquareSum._power(): 4 + 9  → 13
   └─ CubeSum._power():   8 + 27 → 35
```

调用者不需要写 `if type == SquareSum`。它只依赖共同的 `calculate()` 接口。

In [ ]:
try:
    AbstractPowerSum()
except TypeError:
    print("[通过] 抽象类不能直接实例化。")
else:
    raise AssertionError("抽象类本应拒绝实例化")


class IncompletePowerSum(AbstractPowerSum):
    pass


try:
    IncompletePowerSum()
except TypeError:
    print("[通过] 未实现 _power() 的子类也不能实例化。")
else:
    raise AssertionError("不完整子类本应拒绝实例化")

## 6. 四个核心概念放在一起

| 概念 | 关注的问题 | 本例中的证据 | 常见误解 |
|---|---|---|---|
| 封装 | 如何组织状态和行为，并维护合法状态？ | `_power_order`、初始化校验、公共 API | 不只是把变量变成 private |
| 继承 | 子类如何复用并扩展父类？ | `NamedSumPower(SumPower)` | 不是任何代码复用都该继承 |
| 抽象 | 外部需要知道什么能力，不需要知道哪些细节？ | `AbstractPowerSum` 规定 `_power()` 契约 | 不只等于“隐藏代码” |
| 多态 | 相同调用怎样根据对象产生不同行为？ | `calculate()` 动态调用不同 `_power()` | 不只是两个类恰好有同名方法 |

### 封装与抽象的区别

- **封装**更关注边界、状态和不变量：不要让对象随便进入非法状态。
- **抽象**更关注使用者看到的模型：告诉外部“能做什么”，隐藏“具体怎么做”。

### 可选进阶：Python 中的“接口”

Python 没有 `interface` 关键字。`ABC` 可以提供抽象约束和具体方法；`typing.Protocol` 则可以描述“只要具备这些方法，就符合这种能力”。`Protocol` 主要服务于编辑器和静态类型检查器，其类型注解默认不会在运行时强制检查；下面的程序能运行，直接原因是对象确实具有 `calculate()` 方法，也就是 Python 的鸭子类型。

因此，原文所说“接口只能包含抽象方法”是依赖具体语言和版本的简化说法，不应当作所有语言的通则。

In [ ]:
from typing import Protocol


class SupportsCalculate(Protocol):
    def calculate(self, x, y):
        ...


def use_calculator(calculator: SupportsCalculate, x, y):
    return calculator.calculate(x, y)


class PlainAdder:
    def calculate(self, x, y):
        return x + y


assert use_calculator(SquareSum(), 2, 3) == 13
assert use_calculator(PlainAdder(), 2, 3) == 5
print("SquareSum 与 PlainAdder 都具备 calculate 能力。")

## 7. 函数还是类？

| 情况 | 通常优先选择 |
|---|---|
| 一次简单、无状态的计算 | 函数 |
| 配置作为参数传入就很清楚 | 函数 |
| 同一份配置会反复使用 | 类 |
| 多个相关行为需要围绕同一状态协作 | 类 |
| 多个实现需要遵守共同契约 | 抽象类或 Protocol |
| 子类型确实满足 is-a 关系 | 可以考虑继承 |

本例中，`sum_power(2, 3, 4)` 是小任务的最简方案；当幂次需要长期保存、对象需要更多相关行为，或者算法类型越来越多时，`SumPower(4)` 与抽象多态设计才显示优势。

## 8. 联系现有代码：把问候函数重构成对象

你现有的 `lec2_hello_demo.py` 使用一个课程变量和一个问候函数。若课程对象以后还要保存教师、学生和作业等状态，可以重构为类。当前只有一次简单问候时，原来的函数仍然更轻量。

In [ ]:
class Course:
    def __init__(self, course_name):
        if not isinstance(course_name, str) or not course_name.strip():
            raise ValueError("course_name 不能为空")
        self._course_name = course_name.strip()

    @property
    def course_name(self):
        return self._course_name

    def greet(self, student_name):
        return f"你好，{student_name}！欢迎学习 {self.course_name}"


ds2004 = Course("DS2004")
message = ds2004.greet("Ada")

print(message)
assert message == "你好，Ada！欢迎学习 DS2004"

## 9. 动手练习

### 练习 1：增加四次幂类型

先不要看下一格答案。根据 `SquareSum` 和 `CubeSum`，在你自己的新代码单元格中完成：

```python
class FourthPowerSum(AbstractPowerSum):
    def _power(self, x):
        # 在这里实现
        ...

assert FourthPowerSum().calculate(2, 3) == 97
assert FourthPowerSum().calculate(-2, 3) == 97
```

第二个断言也说明：负数的偶次幂是正数。

### 练习 1 参考答案

完成自己的版本后再运行下面的单元格。

In [ ]:
class FourthPowerSum(AbstractPowerSum):
    def _power(self, x):
        return x ** 4


fourth_power_sum = FourthPowerSum()

assert fourth_power_sum.calculate(2, 3) == 97
assert fourth_power_sum.calculate(-2, 3) == 97
print("四次幂之和：", fourth_power_sum.calculate(2, 3))

### 练习 2：验证扩展后的多态

把新类型放入列表。注意循环本身完全不需要修改。

In [ ]:
extended_calculators = [
    SquareSum(),
    CubeSum(),
    FourthPowerSum(),
]

extended_results = [
    calculator.calculate(2, 3)
    for calculator in extended_calculators
]

print(extended_results)
assert extended_results == [13, 35, 97]

### 概念自测

请先口头回答，再展开答案：

1. 创建对象时，哪个方法会自动运行？
2. 为什么 `square_sum` 和 `cube_sum` 可以保存不同幂次？
3. 为什么相同的 `calculate()` 调用能产生 `13` 和 `35`？
4. `_power_order` 在 Python 中是否真正禁止外部访问？
5. 只计算一次 `x ** n + y ** n` 时，通常应先选择函数还是类？

<details>
<summary>点击查看参考答案</summary>

1. `__init__`。
2. 它们是两个独立对象，每个对象都有自己的实例状态。
3. `calculate()` 中的 `self._power()` 会根据对象实际类型动态选择重写后的实现。
4. 不会；单下划线只是非公开 API 的命名约定。
5. 通常先选择函数，因为它更直接。

</details>

## 10. 最终自动检查

下面的断言覆盖本 Notebook 的主要学习成果。任何断言失败，都会指出前面某一部分需要重新检查。

In [ ]:
assert sum_squares(2, 3) == 13, "函数版平方和错误"
assert sum_power(2, 3, 5) == 275, "通用函数版错误"
assert SumPower(4).calculate(2, 3) == 97, "保存状态的类版本错误"
assert NamedSumPower(3, "立方和").calculate(2, 3) == 35, "继承版本错误"
assert [c.calculate(2, 3) for c in extended_calculators] == [13, 35, 97], "多态结果错误"
assert Course("DS2004").greet("Ada") == "你好，Ada！欢迎学习 DS2004", "Course 重构错误"

print("[通过] 所有核心测试通过：Notebook 可以从头到尾运行。")

## 一句话总结

- **函数**把一段计算规则命名并复用；
- **对象**把状态和相关行为放在一起；
- **封装**维护对象边界和合法状态；
- **继承**表达合理的父子类型关系；
- **抽象**规定外部需要的能力；
- **多态**让同一接口根据对象类型执行不同实现。

最后请执行一次 **Restart Kernel and Run All Cells**。如果最终看到 `[通过]` 提示，就说明所有示例在干净环境中能够协同工作。